### MIMII Audio to Mel Spectrogram

Place this script anywhere. Define root as your MIMII dataset folder.

your directory tree should looks like:

```
root
  ├──fan
  │   ├──id_00
  │   │    ├──normal
  │   │    └──abnormal
  │   ├──id_02
  │   │    ├──normal
  │   │    └──abnormal
  │   ├──id_04
  │   │    ├──normal
  │   │    └──abnormal
  │   └──id_06
  │        ├──normal
  │        └──abnormal
  │
  ├──valve
      .
      .
      .
 (same for all 4 machine types)
 ```

In [1]:
import glob, os
import librosa 
import numpy as np
import cv2
from scipy.io import wavfile
from matplotlib import pyplot as plt
import wave

In [2]:
# define your own dataset root

root = '../../tmp/dcase-2020/'
output_root = '../../tmp/dcase-2020-spectrogram/'

In [3]:
# tuple of (input path, output path)
cats = [('data_fan/fan', 'fan'),
		('data_valve/valve', 'valve'),
		('data_pump/pump', 'pump'),
		('data_slider/slider', 'slider'),
		('data_ToyCar/ToyCar', 'ToyCar'),
		('data_ToyConveyor/ToyConveyor', 'ToyConveyor')]
# in every category, there are these subcategories
subs = ['test', 'train']
# labels of (input, output)
labels = [('normal', 'normal'), ('anomaly', 'abnormal')]

In [4]:
def get_paths(cat, sub, label):
    path = root + cat[0] + '/' + sub + '/'
    targetpath = output_root + cat[1] + '/' + sub + '/' + label[1] + '/'
    
    print(path)
    print(targetpath)
    
    # if not os.path.exists(path):
    #     raise FileNotFoundError(f"Path not found: {path}")
    if not os.path.exists(targetpath):
        print("new target path created.")
        os.makedirs(targetpath)

    return path, targetpath

In [5]:
def scale_minmax(X, smin=0.0, smax=1.0):
    X_std = (X - X.min()) / (X.max() - X.min())
    X_scaled = X_std * (smax - smin) + smin
    return X_scaled

def save_mel_img(fname, oname, targetpath, hop_length=512, n_mels=128, fmax=8000, three_channel=True):
    y, sr = librosa.load(fname, sr=None)
    S = librosa.feature.melspectrogram(y=y, 
                                       sr=sr, 
                                       n_mels=n_mels, 
                                       n_fft=hop_length*2, 
                                       hop_length=hop_length, 
                                       fmax=fmax,
                                       power=2.0 # for power spectrogram
                                      )
    S = librosa.power_to_db(S, ref=np.max) # convert to dB scale

    # convert (a,b) to (a, b, 1) for grayscale image


    if three_channel:
        delta1 = librosa.feature.delta(S)
        delta2 = librosa.feature.delta(S, order=2)
        # logMel + delta + delta-delta = 3 channels
        img = np.stack([ scale_minmax(mat, 0, 255).astype(np.uint8) for mat in [S, delta1, delta2]], axis=-1)
    else:
        img = scale_minmax(S, 0, 255).astype(np.uint8)
    img = np.flip(img, axis=0)
    
    # plt.figure(figsize=(4,2))
    # plt.imshow(img)
    # plt.show()
    
    savename = targetpath + oname + '.png'
    # print(savename, S.shape, y.shape[0])
    cv2.imwrite(savename, img)

In [6]:
%%time
import re


for label in labels:
    pattern = re.compile(rf"^{label[0]}_(.*)\.wav$")
    for cat in cats:
        for sub in subs:
            path, targetpath = get_paths(cat, sub, label)
            fnames = glob.glob(path+'*.wav')
            # print("folder size:", len(fnames))
            print(f"Processing {len(fnames)} files in {path}...")

            for fname in fnames:
                m = pattern.match(os.path.basename(fname))
                if m:
                    save_mel_img(fname, m.group(1), targetpath, three_channel=False)

../../tmp/dcase-2020/data_fan/fan/test/
../../tmp/dcase-2020-spectrogram/fan/test/normal/
new target path created.
Processing 1875 files in ../../tmp/dcase-2020/data_fan/fan/test/...


/home/f74134118/anaconda3/envs/mamba-ad/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


../../tmp/dcase-2020/data_fan/fan/train/
../../tmp/dcase-2020-spectrogram/fan/train/normal/
new target path created.
Processing 3675 files in ../../tmp/dcase-2020/data_fan/fan/train/...
../../tmp/dcase-2020/data_valve/valve/test/
../../tmp/dcase-2020-spectrogram/valve/test/normal/
new target path created.
Processing 879 files in ../../tmp/dcase-2020/data_valve/valve/test/...
../../tmp/dcase-2020/data_valve/valve/train/
../../tmp/dcase-2020-spectrogram/valve/train/normal/
new target path created.
Processing 3291 files in ../../tmp/dcase-2020/data_valve/valve/train/...
../../tmp/dcase-2020/data_pump/pump/test/
../../tmp/dcase-2020-spectrogram/pump/test/normal/
new target path created.
Processing 856 files in ../../tmp/dcase-2020/data_pump/pump/test/...
../../tmp/dcase-2020/data_pump/pump/train/
../../tmp/dcase-2020-spectrogram/pump/train/normal/
new target path created.
Processing 3349 files in ../../tmp/dcase-2020/data_pump/pump/train/...
../../tmp/dcase-2020/data_slider/slider/test/
..